In [ ]:
from typing import List
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Transformers

## Создание модели и предсказание следующего токена

In [ ]:
def move_to_device(inputs, device):
    for k, v in inputs.items():
        inputs[k] = v.to(device)
    return inputs

In [ ]:
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")


text = "This is a sample text"

inputs = tokenizer(text, return_tensors="pt")
inputs = move_to_device(inputs, device)

outputs = model(**inputs)
logits = outputs.logits
next_token_idx = logits[0][-1].argmax().item()


next_token = tokenizer.decode([next_token_idx])

assert next_token.strip() == "file"

## Generate

Генерация с:
* Температурой - 0.9
* Top-K - 20
* Frequency Penalty - 1.2
* максимальное число новых токенов - 10

In [ ]:
text = "This is still a sample text, but"
inputs = tokenizer(text, return_tensors="pt")
inputs = move_to_device(inputs, device)

results = []
for i in range(10):
    gens = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.9,
        top_k=20,
        repetition_penalty=1.2,
        max_new_tokens=10,
        pad_token_id=tokenizer.eos_token_id
    )
    results.append(tokenizer.decode(gens[0, inputs.input_ids.size(1):]))

assert len(set(results)) > 1

## Generate Batched

Жадная генерация, но с батчами (паддинги)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2", padding_side="left")
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
texts = ["This is a sample text", "I'm really tired and this is just about"]
inputs = tokenizer(texts, return_tensors="pt", padding=True)
inputs = move_to_device(inputs, device)

batched_generations: List[str] = []
single_generations: List[str] = []

generations_batched = model.generate(
    **inputs,
    max_new_tokens=10
)

batched_generations = tokenizer.batch_decode(generations_batched[:, inputs.input_ids.size(1):])
for text in texts:
    inputs = tokenizer(text, return_tensors="pt")
    inputs = move_to_device(inputs, device)
    g = model.generate(**inputs, max_new_tokens=10)
    single_generations.append(tokenizer.decode(g[0, inputs.input_ids.size(1):]))

assert len(batched_generations) == 2 and len(single_generations) == 2
for s, b in zip(batched_generations, single_generations):
    assert s == b

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


# KV Cache

Сравнить время генерации с KV_cache И без него.
2. Подсчитать скорость генерации 1 токена с и без kv cache, сказать, какая техника быстрее и почему.

In [ ]:
import time
import numpy as np

text = """
Lorem ipsum dolor sit amet, consectetur adipiscing elit. Vestibulum lorem justo, semper dignissim ipsum vitae, sollicitudin aliquet eros. Duis id ultricies erat. Vivamus commodo auctor massa ut mollis. Maecenas lacinia tempus orci, imperdiet ullamcorper felis accumsan et. Etiam mattis neque diam, at egestas nunc eleifend id. Fusce tristique orci nec sollicitudin elementum. Nullam dui est, feugiat ac pellentesque at, posuere non massa.

Suspendisse accumsan ullamcorper dolor sed dictum. Mauris quis varius felis, quis gravida odio. Vestibulum diam arcu, aliquet convallis congue non, rutrum non turpis. Fusce vel orci ac diam suscipit lacinia. Curabitur maximus orci a dui gravida, accumsan convallis libero ornare. Phasellus dapibus, sapien pulvinar lacinia dictum, massa lacus scelerisque tellus, eu porta dolor eros vitae ex. Maecenas maximus, urna id pharetra dictum, dolor lorem sollicitudin ipsum, sit amet vestibulum orci felis quis leo. Pellentesque vel ligula ut urna eleifend condimentum nec et sem. Integer ligula nunc, rutrum ultricies urna et, congue suscipit lectus.
""".strip()

for use_cache in (True, False):
  times = []
  for _ in range(10):
    start = time.time()
    inputs = tokenizer(text, return_tensors="pt")
    inputs = move_to_device(inputs, device)
    model.generate(**inputs, use_cache=use_cache, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
    times.append(time.time() - start)
  print(f"{'with' if use_cache else 'without'} KV caching: {round(np.mean(times), 3)} +- {round(np.std(times), 3)} seconds")

with KV caching: 0.786 +- 0.016 seconds
without KV caching: 1.138 +- 0.003 seconds


# Скоринг, Perplixity

1. Посчитать вероятность **text**
2. Посчитать перплексию **text**

In [ ]:
text = "<|endoftext|>I'm so very tired of this<|endoftext|>"

inputs = tokenizer(text, return_tensors="pt")
inputs = move_to_device(inputs, device)
input_ids = inputs.input_ids

with torch.no_grad():
    logits = model(**inputs).logits[0] # seq_len, vocab_size
    logits = logits[:-1] # т.к. для последнего токена ничего не генерируем
    probs = torch.log_softmax(logits, dim=1) # превращаем в вероятноси
    targets = input_ids[0, 1:] # сдвигаем, т.к. 0й токен мы не предсказываем, а предсказывем сразу следующий, т.е. 1й
    text_probs = torch.gather(probs, 1, targets.unsqueeze(1)).squeeze(1)
    text_P = text_probs.sum().exp()
    ppl = (-text_probs.mean()).exp()

print(text_P)
print(ppl)

Beginning of sentence (BOS) token = `<|endoftext|>`
End of sentence (EOS) token  = `<|endoftext|>`
tensor(2.1782e-14, device='cuda:0')
tensor(51.0197, device='cuda:0')
